In [1]:
import pandas as pd
import numpy as np
from src.utils import get_data_directory, get_submissions_directory
from src.evaluation import compute_brier_score

#### Arguments

In [2]:
YEAR = 2024
SUBMISSION_FILE = "SampleSubmissionStage1.csv"

#### Prepare NCAA Tournament Data

In [3]:
df_men_tourneys = pd.read_csv(get_data_directory() / "MNCAATourneyCompactResults.csv")
df_women_tourneys = pd.read_csv(get_data_directory() / "WNCAATourneyCompactResults.csv")

df_tourneys = pd.concat((df_men_tourneys, df_women_tourneys))

In [4]:
# isolate only the year in question
df_tourney_year = df_tourneys[df_tourneys["Season"] == YEAR].copy()

# add column with the 'Result' in terms of the lower ID team -> 1 if lower ID team won, 0 if higher ID team won
df_tourney_year["Result"] = np.where(df_tourney_year["WTeamID"] < df_tourney_year["LTeamID"], 1, 0)

# add columns for lowerID and higherID
df_tourney_year["lowerID"] = df_tourney_year[["WTeamID", "LTeamID"]].min(axis=1)
df_tourney_year["higherID"] = df_tourney_year[["WTeamID", "LTeamID"]].max(axis=1)

# drop unused columns
df_tourney_year = df_tourney_year[["lowerID", "higherID", "Result"]]
print(df_tourney_year.duplicated().sum())

0


In [5]:
df_tourney_year.head()

,lowerID,higherID,Result
2451,1161,1438,1
2452,1224,1447,0
2453,1129,1160,0
2454,1212,1286,1
2455,1112,1253,1


#### Prepare Submission Data

In [6]:
df_submission = pd.read_csv(get_submissions_directory() / SUBMISSION_FILE)

df_submission[["Season", "lowerID", "higherID"]] = df_submission["ID"].str.split("_", expand=True)
df_submission[["Season", "lowerID", "higherID"]] = df_submission[["Season", "lowerID", "higherID"]].apply(pd.to_numeric)

# isolate only the year in question
df_submission_year = df_submission[df_submission["Season"] == YEAR].copy()

# drop unused columns
df_submission_year = df_submission_year[["lowerID", "higherID", "Pred"]]
print(df_submission_year.duplicated().sum())

0


In [7]:
df_submission_year.head()

,lowerID,higherID,Pred
377147,1101,1102,0.5
377148,1101,1103,0.5
377149,1101,1104,0.5
377150,1101,1105,0.5
377151,1101,1106,0.5


#### Combine DataFrames

In [8]:
len(df_tourney_year)

134

In [9]:
len(df_submission_year)

129961

In [10]:
# merge the two dataframes on lowerID and higherID
df_merged = pd.merge(df_tourney_year, df_submission_year, on=["lowerID", "higherID"], how="left")
missing_pred = df_merged["Pred"].isna().sum()

if missing_pred > 0:
    print(f"{missing_pred} rows have no predictions in the merged DataFrame.")
print(f"Merged dataframe has {len(df_merged)} rows")

Merged dataframe has 134 rows


In [11]:
df_merged.head()

,lowerID,higherID,Result,Pred
0,1161,1438,1,0.5
1,1224,1447,0,0.5
2,1129,1160,0,0.5
3,1212,1286,1,0.5
4,1112,1253,1,0.5


#### Calculate Brier Score

In [12]:
# Calculate Brier Score on entire DataFrame with predictions denoted as 'Pred' and actual results as 'Result'
brier_score = np.mean((df_merged["Pred"] - df_merged["Result"]) ** 2)
print(f"Brier Score: {brier_score:.6f}")

Brier Score: 0.250000


#### Test with perfect predictions

In [13]:
# set all predictions to the actual results
df_merged["Pred"] = df_merged["Result"]

brier_score = np.mean((df_merged["Pred"] - df_merged["Result"]) ** 2)
print(f"Brier Score: {brier_score:.6f}")

Brier Score: 0.000000


#### Test with library function

In [14]:
print(f"Brier Score: {compute_brier_score(SUBMISSION_FILE, YEAR):.6f}")

Brier Score: 0.250000
